# A6 style transfer — core

Adapted from University of Michigan EECS 498/598, Deep Learning for Computer Vision (Winter 2022), by Justin Johnson and course staff.

**Static validation only; not executed.** Enable the run switches explicitly; generated files go to `core_results/`.

In [ ]:
from pathlib import Path
import os
import sys
import torch
import matplotlib.pyplot as plt
candidates = []
for base in (Path.cwd(), *Path.cwd().parents,
             Path('/content/drive/MyDrive/umich-dlcv-wi2022')):
    candidates.extend((base, base / 'assignments' / 'a6'))
ASSIGNMENT_DIR = next((p.resolve() for p in candidates if (p / 'style_transfer_core.py').is_file()), None)
if ASSIGNMENT_DIR is None:
    raise FileNotFoundError('Open from the project/assignment directory, or mount the canonical Colab Drive project first.')
loaded_helpers = sys.modules.get('eecs598')
if loaded_helpers is not None:
    loaded_path = Path(loaded_helpers.__file__).resolve().parent
    if loaded_path != (ASSIGNMENT_DIR / 'eecs598').resolve():
        raise RuntimeError('Restart the kernel before switching assignments: eecs598 is already loaded from another assignment.')
sys.path.insert(0, str(ASSIGNMENT_DIR))
os.chdir(ASSIGNMENT_DIR)
from eecs598.utils import reset_seed
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = ASSIGNMENT_DIR / 'core_results' / 'style_transfer'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_OVERFIT = False
ALLOW_DOWNLOAD = False
reset_seed(0)
plt.rcParams['image.cmap'] = 'gray'
print('Outputs:', OUTPUT_DIR)

In [ ]:
import torchvision
import torchvision.transforms as T
import PIL.Image
from a6_helper_core import preprocess, deprocess, get_zero_one_masks
from style_transfer_core import content_loss, gram_matrix, style_loss, tv_loss, guided_gram_matrix, guided_style_loss
to_float_cuda = torch.cuda.FloatTensor if DEVICE.type == 'cuda' else torch.FloatTensor
RUN_STYLE_TRANSFER = RUN_EVALUATION

## Content/style assets and frozen SqueezeNet
Use retained images in `styles/` and `images/`. Set `RUN_EVALUATION` in setup to optimize pixels; the pretrained model remains fixed. Three styles, feature inversion, and spatially guided transfer are included.

In [ ]:
if RUN_STYLE_TRANSFER:
    required = ['styles/tubingen.jpg', 'styles/composition_vii.jpg', 'styles/the_scream.jpg', 'styles/starry_night.jpg',
                'images/fig2_content.jpg', 'images/fig2_content_nosky.jpg', 'images/fig2_content_sky.jpg',
                'images/fig2_style1.jpg', 'images/fig2_style1_nosky.jpg', 'images/fig2_style2.jpg', 'images/fig2_style2_sky.jpg']
    missing = [name for name in required if not (ASSIGNMENT_DIR / name).is_file()]
    if missing:
        raise FileNotFoundError('Retained course style/mask images required: ' + ', '.join(missing))
    cnn = torchvision.models.squeezenet1_1(weights=torchvision.models.SqueezeNet1_1_Weights.DEFAULT).features.to(DEVICE).eval()
    for parameter in cnn.parameters():
        parameter.requires_grad_(False)

In [ ]:
def extract_features(x, cnn):
    features = []
    prev_feat = x
    for i, module in enumerate(cnn._modules.values()):
        next_feat = module(prev_feat)
        features.append(next_feat)
        prev_feat = next_feat
    return features

def features_from_img(imgpath, imgsize):
    img = preprocess(PIL.Image.open(imgpath), size=imgsize)
    img_var = img.type(to_float_cuda)
    return (extract_features(img_var, cnn), img_var)

In [ ]:
def style_transfer(content_image, style_image, image_size, style_size, content_layer, content_weight, style_layers, style_weights, tv_weight, init_random=False, save_image=False, result_filename=None):
    content_img = preprocess(PIL.Image.open(content_image), size=image_size).type(to_float_cuda)
    feats = extract_features(content_img, cnn)
    content_target = feats[content_layer].clone()
    style_img = preprocess(PIL.Image.open(style_image), size=style_size).type(to_float_cuda)
    feats = extract_features(style_img, cnn)
    style_targets = []
    for idx in style_layers:
        style_targets.append(gram_matrix(feats[idx].clone()))
    if init_random:
        img = torch.Tensor(content_img.size()).uniform_(0, 1).type(to_float_cuda)
    else:
        img = content_img.clone().type(to_float_cuda)
    img.requires_grad_()
    initial_lr = 3.0
    decayed_lr = 0.1
    decay_lr_at = 180
    optimizer = torch.optim.Adam([img], lr=initial_lr)
    f, axarr = plt.subplots(1, 2)
    axarr[0].axis('off')
    axarr[1].axis('off')
    axarr[0].set_title('Content Source Img.')
    axarr[1].set_title('Style Source Img.')
    axarr[0].imshow(deprocess(content_img.cpu()))
    axarr[1].imshow(deprocess(style_img.cpu()))
    plt.show()
    plt.figure()
    for t in range(200):
        if t < 190:
            img.data.clamp_(-1.5, 1.5)
        optimizer.zero_grad()
        feats = extract_features(img, cnn)
        c_loss = content_loss(content_weight, feats[content_layer], content_target)
        s_loss = style_loss(feats, style_layers, style_targets, style_weights)
        t_loss = tv_loss(img, tv_weight)
        loss = c_loss + s_loss + t_loss
        loss.backward()
        if t == decay_lr_at:
            optimizer = torch.optim.Adam([img], lr=decayed_lr)
        optimizer.step()
        if t % 100 == 0:
            print('Iteration {}'.format(t))
            plt.axis('off')
            plt.imshow(deprocess(img.data.cpu()))
            plt.show()
    print('Iteration {}'.format(t))
    plt.axis('off')
    plt.imshow(deprocess(img.data.cpu()))
    if save_image:
        plt.savefig(os.path.join(str(OUTPUT_DIR), result_filename))
    plt.show()

## Composition VII

In [ ]:
if RUN_STYLE_TRANSFER:
    params1 = {'content_image': 'styles/tubingen.jpg', 'style_image': 'styles/composition_vii.jpg', 'image_size': 192, 'style_size': 512, 'content_layer': 3, 'content_weight': 0.05, 'style_layers': (1, 4, 6, 7), 'style_weights': (20000, 500, 12, 1), 'tv_weight': 0.05, 'save_image': True, 'result_filename': 'style_transfer_result.jpg'}
    style_transfer(**params1)

## The Scream

In [ ]:
if RUN_STYLE_TRANSFER:
    params2 = {'content_image': 'styles/tubingen.jpg', 'style_image': 'styles/the_scream.jpg', 'image_size': 192, 'style_size': 224, 'content_layer': 3, 'content_weight': 0.03, 'style_layers': [1, 4, 6, 7], 'style_weights': [200000, 800, 12, 1], 'tv_weight': 0.02, 'save_image': True, 'result_filename': 'scream_style_transfer.jpg'}
    style_transfer(**params2)

## Starry Night

In [ ]:
if RUN_STYLE_TRANSFER:
    params3 = {'content_image': 'styles/tubingen.jpg', 'style_image': 'styles/starry_night.jpg', 'image_size': 192, 'style_size': 192, 'content_layer': 3, 'content_weight': 0.06, 'style_layers': [1, 4, 6, 7], 'style_weights': [300000, 1000, 15, 3], 'tv_weight': 0.02, 'save_image': True, 'result_filename': 'starry_style_transfer.jpg'}
    style_transfer(**params3)

## Feature inversion

In [ ]:
if RUN_STYLE_TRANSFER:
    params_inv = {'content_image': 'styles/tubingen.jpg', 'style_image': 'styles/starry_night.jpg', 'image_size': 192, 'style_size': 192, 'content_layer': 3, 'content_weight': 0.06, 'style_layers': [1, 4, 6, 7], 'style_weights': [0, 0, 0, 0], 'tv_weight': 0.02, 'init_random': True, 'save_image': True, 'result_filename': 'feature_inversion_result.jpg'}
    style_transfer(**params_inv)

## Spatially guided transfer

In [ ]:
def get_image_masks(img_paths, img_size):
    masks = []
    for path in img_paths:
        masks.append(get_zero_one_masks(PIL.Image.open(path), img_size).type(to_float_cuda))
    return torch.cat(masks, 0)

def extract_layer_masks(features, masks):
    layer_masks = []
    for feat in features:
        feat_height, feat_width = (feat.shape[-2], feat.shape[-1])
        feat_transform = T.Resize((feat_height, feat_width))
        feat_masks = feat_transform(masks)
        layer_masks.append(feat_masks)
    return layer_masks

def extract_regional_features(x, cnn, R=2):
    features = []
    prev_feat = x
    for i, module in enumerate(cnn._modules.values()):
        next_feat = module(prev_feat)
        features.append(next_feat.unsqueeze(1).repeat(1, R, 1, 1, 1))
        prev_feat = next_feat
    return features

def guided_style_transfer(content_image, style_images, image_size, style_size, content_layer, content_weight, content_masks, style_layers, style_weights, style_masks, tv_weight, init_random=False, save_image=False, result_filename=None):
    content_img = preprocess(PIL.Image.open(content_image), size=image_size).type(to_float_cuda)
    feats = extract_regional_features(content_img, cnn)
    content_target = feats[content_layer].clone()
    style1_img, style2_img = [preprocess(PIL.Image.open(style_image), size=style_size).type(to_float_cuda) for style_image in style_images]
    style1_feats = extract_features(style1_img, cnn)
    style2_feats = extract_features(style2_img, cnn)
    content_masks = extract_layer_masks(style1_feats, content_masks)
    feats = []
    for i in range(len(style1_feats)):
        style_feats = torch.stack((style1_feats[i], style2_feats[i]), dim=1)
        feats.append(style_feats)
    style_masks = extract_layer_masks(feats, style_masks)
    style_targets = []
    for idx in style_layers:
        style_targets.append(guided_gram_matrix(feats[idx].clone(), style_masks[idx]))
    if init_random:
        img = torch.Tensor(content_img.size()).uniform_(0, 1).type(to_float_cuda)
    else:
        img = content_img.clone().type(to_float_cuda)
    img.requires_grad_()
    initial_lr = 3.0
    decayed_lr = 0.1
    decay_lr_at = 180
    optimizer = torch.optim.Adam([img], lr=initial_lr)
    f, axarr = plt.subplots(1, 3)
    axarr[0].axis('off')
    axarr[1].axis('off')
    axarr[2].axis('off')
    axarr[0].set_title('Content Source Img.')
    axarr[1].set_title('Style1 Source Img.')
    axarr[2].set_title('Style2 Source Img.')
    axarr[0].imshow(deprocess(content_img.cpu()))
    axarr[1].imshow(deprocess(style1_img.cpu()))
    axarr[2].imshow(deprocess(style2_img.cpu()))
    plt.show()
    plt.figure()
    for t in range(200):
        if t < 190:
            img.data.clamp_(-1.5, 1.5)
        optimizer.zero_grad()
        feats = extract_regional_features(img, cnn)
        c_loss = content_loss(content_weight, feats[content_layer], content_target)
        s_loss = guided_style_loss(feats, style_layers, style_targets, style_weights, content_masks)
        t_loss = tv_loss(img, tv_weight)
        loss = c_loss + s_loss + t_loss
        loss.backward()
        if t == decay_lr_at:
            optimizer = torch.optim.Adam([img], lr=decayed_lr)
        optimizer.step()
        if t % 100 == 0:
            print('Iteration {}'.format(t))
            plt.axis('off')
            plt.imshow(deprocess(img.data.cpu()))
            plt.show()
    print('Iteration {}'.format(t))
    plt.axis('off')
    plt.imshow(deprocess(img.data.cpu()))
    if save_image:
        plt.savefig(os.path.join(str(OUTPUT_DIR), result_filename))
    plt.show()

In [ ]:
if RUN_STYLE_TRANSFER:
    image_folder_path = os.path.join(str(ASSIGNMENT_DIR), 'images')
    image_size = (192, 288)
    content_sky_mask_path = os.path.join(image_folder_path, 'fig2_content_sky.jpg')
    content_nosky_mask_path = os.path.join(image_folder_path, 'fig2_content_nosky.jpg')
    content_masks = get_image_masks([content_nosky_mask_path, content_sky_mask_path], image_size).unsqueeze(0)
    style1_nosky_mask_path = os.path.join(image_folder_path, 'fig2_style1_nosky.jpg')
    style2_sky_mask_path = os.path.join(image_folder_path, 'fig2_style2_sky.jpg')
    style_masks = get_image_masks([style1_nosky_mask_path, style2_sky_mask_path], image_size).unsqueeze(0)
    content_image_path = os.path.join(image_folder_path, 'fig2_content.jpg')
    style1_image_path = os.path.join(image_folder_path, 'fig2_style1.jpg')
    style2_image_path = os.path.join(image_folder_path, 'fig2_style2.jpg')

In [ ]:
if RUN_STYLE_TRANSFER:
    params_inv = {'content_image': content_image_path, 'style_images': [style1_image_path, style2_image_path], 'image_size': (192, 288), 'style_size': (192, 288), 'content_layer': 3, 'content_weight': 0.02, 'content_masks': content_masks, 'style_layers': [1, 4, 6, 7], 'style_weights': [300000, 1000, 15, 3], 'style_masks': style_masks, 'tv_weight': 0.02, 'init_random': True, 'save_image': True, 'result_filename': 'spatial_style_transfer_result.jpg'}
    guided_style_transfer(**params_inv)